# Simple Neural Network With TensorFlow

In this lab, we'll build two neural networks with TensorFlow's core API (`tf.Variable`, `tf.Module`, `tf.GradientTape`, `tf.data`), without a Keras model: a network without a hidden layer first, then a multilayer perceptron (MLP). The Keras lab built the same networks with `compile` and `fit`: this one writes by hand what `fit` does.

The data is MNIST, which the Keras lab explored: 28 $\times$ 28 pixel grayscale images of the digits 0 to 9. Each image comes with a label, the digit the image is supposed to represent.

The steps:

1. Get started with TensorFlow tensors
2. Recover the coefficients of a line by gradient descent, with `tf.GradientTape`
3. Load and prepare the data, set a validation set aside
4. Write a network without a hidden layer, its loss, its accuracy and its `tf.data` pipeline
5. Train it by gradient descent written by hand, then with a Keras optimizer
6. Compile the training step with `tf.function`, and count its tracings
7. Move on to an MLP and regularize it
8. Follow its training runs in TensorBoard, choose its learning rate on the validation set
9. Evaluate it on the test set, save it and restore it

This notebook needs no GPU: each training run takes from a few seconds to a few minutes on Colab's CPU.

## Imports

In [ ]:
import time
from collections.abc import Callable

import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# Same draws from one run to the next: initial weights, shuffling
keras.utils.set_random_seed(0)

TensorFlow provides the tensors, the gradients and the data pipeline; Keras, its MNIST dataset, its loss function, its optimizers, and the seed of the random draws (`keras.utils.set_random_seed`), so that your numbers are those of the solutions. [`matplotlib`](https://matplotlib.org/) and [`numpy`](https://numpy.org/) are used for the plots and simple array operations.

## Getting Started With Tensors

On Colab, TensorFlow is already installed. On your machine, install it with `pip install tensorflow`.

Before building networks, let's handle TensorFlow tensors, whose API is close to NumPy's. The operations that fail raise a `tf.errors.InvalidArgumentError`: catch it with `try`/`except` to display its message.

1. Create a tensor `a` of shape (2, 3) holding the integers from 0 to 5 (`tf.range`, then `tf.reshape`). What is its `dtype`? Convert it to `float32` with `tf.cast`, then create a tensor `b` of shape (3,) filled with 1s (`tf.ones`).
2. Compute `a + b` and the matrix product `a @ tf.ones((3, 2))`. What are the shapes of the results? Why does `a + tf.ones(2)` fail? And `a @ b`?
3. Extract the second column of `a`, then its last row. Add a dimension of size 1 in front of `a` with `tf.expand_dims`: what shape do you get? What is such a dimension for with a model?
4. Convert `a` to a NumPy array (`.numpy()`), then back to a tensor (`tf.convert_to_tensor`). Convert `np.ones(3)` the same way: what `dtype` do you get, and what happens if you add it to `a`? Finally, list the GPUs TensorFlow sees (`tf.config.list_physical_devices`).

**Useful documentation**

- [`tf.range`](https://www.tensorflow.org/api_docs/python/tf/range)
- [`tf.reshape`](https://www.tensorflow.org/api_docs/python/tf/reshape)
- [`tf.ones`](https://www.tensorflow.org/api_docs/python/tf/ones)
- [`tf.cast`](https://www.tensorflow.org/api_docs/python/tf/cast)
- [`tf.convert_to_tensor`](https://www.tensorflow.org/api_docs/python/tf/convert_to_tensor)
- [`tf.expand_dims`](https://www.tensorflow.org/api_docs/python/tf/expand_dims)
- [`tf.config.list_physical_devices`](https://www.tensorflow.org/api_docs/python/tf/config/list_physical_devices)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `tf.reshape(tf.range(6), (2, 3))`, then `tf.cast(a, tf.float32)`
- `a[:, 1]`: the second column; `a[-1]`: the last row
- `tf.expand_dims(a, axis=0)`: a dimension of size 1 at position 0
- `tf.config.list_physical_devices("GPU")`: the list of visible GPUs, empty without a GPU
- An operation that fails, in a `try`:

```python
try:
  a + tf.ones(2)
except tf.errors.InvalidArgumentError as error:
  print(error)
```

A skeleton to copy into the exercise cell, replacing each `...`:

```python
a = tf.reshape(..., (2, 3))
print(a.dtype)
a = tf.cast(a, ...)
b = ...

print((a + b).shape, (a @ tf.ones((3, 2))).shape)
try:
  ...  # a + tf.ones(2), then a @ b in a second try
except tf.errors.InvalidArgumentError as error:
  print(error)

print(a[:, 1], ...)  # second column, last row
print(tf.expand_dims(a, axis=0).shape)

print(tf.convert_to_tensor(a.numpy()).dtype, tf.convert_to_tensor(np.ones(3)).dtype)
try:
  ...  # a + the tensor created from np.ones(3)
except tf.errors.InvalidArgumentError as error:
  print(error)
print(tf.config.list_physical_devices(...))
```

### Solution

In [ ]:
a = tf.reshape(tf.range(6), (2, 3))
print(a, a.dtype)
a = tf.cast(a, tf.float32)
b = tf.ones(3)

# b is broadcast over each row of a
print(a + b, (a + b).shape)
# Matrix product: (2, 3) @ (3, 2) -> (2, 2)
print(a @ tf.ones((3, 2)))

try:
  a + tf.ones(2)
except tf.errors.InvalidArgumentError as error:
  print("a + tf.ones(2):", error)
try:
  a @ b
except tf.errors.InvalidArgumentError as error:
  print("a @ b:", error)

print(a[:, 1])  # second column
print(a[-1])  # last row
print(tf.expand_dims(a, axis=0).shape)  # batch dimension of size 1

print(type(a.numpy()), tf.convert_to_tensor(a.numpy()).dtype)
ones_64 = tf.convert_to_tensor(np.ones(3))
print(ones_64.dtype)
try:
  a + ones_64
except tf.errors.InvalidArgumentError as error:
  print("a + ones_64:", error)
print(tf.config.list_physical_devices("GPU"))

- `tf.range` creates 32-bit integers (`int32`); neural networks compute with floats, `float32` by default.
- `a + b` has shape (2, 3) and `a @ tf.ones((3, 2))` shape (2, 2). For broadcasting, dimensions are compared starting from the last one: they must be equal, or one of them must be 1 or missing. (2, 3) and (3,) are compatible, but not (2, 3) and (2,): it would take the shape (2, 1), `tf.ones((2, 1))`.
- `a @ b` fails too: in TensorFlow, `@` multiplies matrices (at least two dimensions), whereas NumPy accepts a vector. It takes `b` of shape (3, 1), or `tf.linalg.matvec(a, b)`.
- `tf.expand_dims(a, axis=0)` has shape (1, 2, 3). Models process batches, the first dimension being the examples': to predict on a single example, we add a batch dimension of size 1 to it.
- `tf.convert_to_tensor` keeps the NumPy type: `float32` for `a.numpy()`, but `float64` for `np.ones(3)`, NumPy's default type. TensorFlow never converts a type into another by itself: `float32 + float64` raises an error, where NumPy promotes to `float64`. Hence the `tf.cast`, or `.astype("float32")` on the NumPy side, which the preparation of the MNIST images will do.
- Without a GPU, the list is empty and TensorFlow computes on the CPU; with a GPU, it uses it automatically.

## Linear Regression With `GradientTape`

Before writing a neural network, let's train the simplest of models, a line $\hat{y} = wx + b$, by gradient descent written by hand. The networks of this lab will follow the same loop, with more variables.

The next cell creates the data: 100 points around the line $y = 3x + 2$, with a Gaussian noise of standard deviation 1. The goal is to recover $w = 3$ and $b = 2$ from the points alone.

In [ ]:
rng = np.random.default_rng(0)
x = rng.uniform(0, 5, size=100).astype("float32")
y = (3 * x + 2 + rng.normal(0, 1, size=100)).astype("float32")

plt.scatter(x, y, s=10)
plt.xlabel("x")
plt.ylabel("y")
plt.show()

*Create two `tf.Variable`s, `w` and `b`, zero at first. At each of 500 steps, compute under a `tf.GradientTape` the loss, the mean squared error (MSE) $\frac{1}{n} \sum_i (w x_i + b - y_i)^2$, then its gradients with respect to `w` and `b` (`tape.gradient`), and subtract 0.05 times its gradient from each variable (`assign_sub`). Keep each step's loss in a list, plot it, and display `w`, `b` and the final loss. What happens with a learning rate of 0.1?*

**Useful documentation**

- [`tf.Variable`](https://www.tensorflow.org/api_docs/python/tf/Variable)
- [`tf.GradientTape`](https://www.tensorflow.org/api_docs/python/tf/GradientTape)
- [`tf.Variable.assign_sub`](https://www.tensorflow.org/api_docs/python/tf/Variable#assign_sub)
- [`tf.reduce_mean`](https://www.tensorflow.org/api_docs/python/tf/math/reduce_mean)

In [ ]:
# Your code here

### Additional hints

- `tf.Variable(0.0)`: a scalar variable, in `float32` like `x` and `y`
- `tf.reduce_mean((w * x + b - y) ** 2)`: the MSE, computed on the 100 points at once, inside the `with tf.GradientTape() as tape:` block so that the tape records the operations
- `tape.gradient(loss, [w, b])`: the two gradients, in the order of the variables
- `w.assign_sub(learning_rate * dw)`: $w \leftarrow w - \eta \, \partial L / \partial w$, after the tape's block
- `float(loss)`: the loss, as a Python number, to append to the list
- `plt.yscale("log")`: a logarithmic scale, which shows both the first steps and the last ones

Expected: a loss of about 124 at the first step, and below 1 at the end; `w` and `b` close to 3 and 2.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
w = tf.Variable(...)
b = tf.Variable(...)
learning_rate = 0.05
losses = []

for step in range(500):
  with tf.GradientTape() as tape:
    loss = ...
  dw, db = tape.gradient(..., [w, b])
  w.assign_sub(...)
  b.assign_sub(...)
  losses.append(float(loss))

plt.plot(losses)
plt.yscale("log")
plt.xlabel("Step")
plt.ylabel("MSE")
plt.show()
print(f"w = {w.numpy():.2f}, b = {b.numpy():.2f}, MSE = {loss:.2f}")
```

### Solution

In [ ]:
w = tf.Variable(0.0)
b = tf.Variable(0.0)
learning_rate = 0.05
losses = []

for step in range(500):
  # The operations on w and b, recorded by the tape
  with tf.GradientTape() as tape:
    loss = tf.reduce_mean((w * x + b - y) ** 2)  # MSE
  dw, db = tape.gradient(loss, [w, b])
  w.assign_sub(learning_rate * dw)  # w ← w - η ∂L/∂w
  b.assign_sub(learning_rate * db)
  losses.append(float(loss))

plt.plot(losses)
plt.yscale("log")
plt.xlabel("Step")
plt.ylabel("MSE")
plt.title("Loss during gradient descent")
plt.show()
print(f"w = {w.numpy():.2f}, b = {b.numpy():.2f}, MSE = {loss:.2f}")

- The loss goes from 124 at the first step to 1.6 at the second, then settles at 0.94 within about a hundred steps: $w = 2.97$ and $b = 1.99$. These are, to the hundredth, the least squares coefficients of these 100 points (`np.polyfit(x, y, 1)`): gradient descent found the best line, and the gap to 3 and 2 comes from the noise, not from the optimization. The final MSE, close to 1, is the noise's variance, which the line can't explain.
- With a learning rate of 0.1, the loss grows at each step, up to infinity (`inf`): each step overshoots the minimum and moves further away from it on the other side. The maximal learning rate depends on the data: here, the descent diverges above about 0.094.
- Neural networks are trained the same way, with `model.trainable_variables` instead of `[w, b]`, and mini-batches instead of the whole set.

## The Data

The Keras lab explored MNIST: 60,000 training images and 10,000 test images, each a `uint8` array of 28 $\times$ 28 pixels, from 0 (background) to 255 (ink), labels from 0 to 9, and roughly balanced classes. The next cell loads the data, displays its shapes and a few examples.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
print(X_train.shape, X_train.dtype, y_train.shape, X_test.shape)

_, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
  ax.imshow(X_train[i], cmap="gray_r")
  ax.set_title(str(y_train[i]))
  ax.axis("off")
plt.show()

## Preparing the Images

Three transformations prepare the images for the network:

1. Convert them to `float32`: the model's variables are `float32`, and TensorFlow refuses to mix two types.
2. Scale the pixels to the $[0, 1]$ interval rather than between $0$ and $255$.
3. Flatten them: a network without a hidden layer takes one vector per example, of shape `(_, 28²)` rather than `(_, 28, 28)`, where `_` is the number of examples.

The `prepare_images` function applies them to an array or a tensor of images. The images stay stored raw, as `uint8`, four times lighter than as `float32`: the `tf.data` pipeline will prepare them batch by batch, and we'll prepare a whole set at once to evaluate a model on it.

In [ ]:
def prepare_images(images: np.ndarray | tf.Tensor) -> tf.Tensor:
  # (n, 28, 28) uint8, 0 to 255 -> (n, 784) float32, 0 to 1
  images = tf.cast(images, tf.float32) / 255
  return tf.reshape(images, (-1, 28 * 28))


print(prepare_images(X_train[:5]).shape, prepare_images(X_train[:5]).dtype)

## Train / Validation Split

To choose the hyperparameters (learning rate, network size…), we compare models on a **validation** set. The test set is kept for the final evaluation: using it to make choices would make that evaluation too optimistic.

We set the last 10,000 training examples aside for validation:

In [ ]:
X_train, X_val = X_train[:50_000], X_train[50_000:]
y_train, y_val = y_train[:50_000], y_train[50_000:]
print(X_train.shape, X_val.shape, X_test.shape)

## Neural Network Without a Hidden Layer

A neural network without a hidden layer computes $\text{softmax}(XW + b)$: it's **multinomial logistic regression**. `W` is the weight matrix, multiplied by the input, and `b` the bias vector, added next. The softmax turns the 10 scores of each example, its *logits*, into probabilities.

### Weight Initialization

*Complete the `init_weights(n_in, n_out)` function, which returns a `tf.Variable` of shape `(n_in, n_out)` drawn from a normal distribution (`tf.random.normal`), divided by $\sqrt{n_\text{in}}$.*

This division, LeCun initialization, gives the weights a variance of $1 / n_\text{in}$: the output of the layer, a sum of $n_\text{in}$ terms, stays of the same order of magnitude as its inputs, however many there are.

**Useful documentation**

- [`tf.random.normal`](https://www.tensorflow.org/api_docs/python/tf/random/normal)

In [ ]:
def init_weights(n_in: int, n_out: int) -> tf.Variable:
  pass  # Your code here

#### Additional hints

- `tf.random.normal((n_in, n_out))`: a tensor of shape `(n_in, n_out)`, with mean 0 and standard deviation 1
- `n_in**0.5`: the square root of `n_in`
- `tf.Variable(tensor)`: a variable, which TensorFlow will be able to update

Expected: `init_weights(784, 10)` has shape (784, 10) and a standard deviation close to $1 / 28 \approx 0.036$ (`tf.math.reduce_std`).

```python
def init_weights(n_in: int, n_out: int) -> tf.Variable:
  return tf.Variable(tf.random.normal(...) / ...)
```

#### Solution

In [ ]:
def init_weights(n_in: int, n_out: int) -> tf.Variable:
  # Variance 1 / n_in: outputs of the order of 1, whatever n_in
  return tf.Variable(tf.random.normal((n_in, n_out)) / n_in**0.5)


W = init_weights(784, 10)
print(W.shape, float(tf.math.reduce_std(W)))

### Defining the Model

*Complete the `SimpleNN` class: in `__init__`, create `self.W` with `init_weights` and `self.b`, a `tf.Variable` of zeros of shape `(n_outputs,)`; `__call__` returns the logits $XW + b$.*

The model doesn't compute the softmax: the loss function will apply it itself (`from_logits=True`), in a more numerically stable way.

In [ ]:
class SimpleNN(tf.Module):
  def __init__(self, n_inputs: int = 28**2, n_outputs: int = 10) -> None:
    super().__init__()
    # Your code here

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    pass  # Your code here

#### Additional hints

- `tf.Variable(tf.zeros(n_outputs))`: the biases, zero at the start
- `X @ self.W + self.b`: the matrix product, then adding the biases, broadcast over each row

Expected: `SimpleNN()(prepare_images(X_train[:5]))` has shape (5, 10), and `SimpleNN().trainable_variables` holds 2 variables.

```python
class SimpleNN(tf.Module):
  def __init__(self, n_inputs: int = 28**2, n_outputs: int = 10) -> None:
    super().__init__()
    self.W = init_weights(..., ...)
    self.b = tf.Variable(...)

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    return ...
```

#### Solution

In [ ]:
class SimpleNN(tf.Module):
  def __init__(self, n_inputs: int = 28**2, n_outputs: int = 10) -> None:
    super().__init__()
    self.W = init_weights(n_inputs, n_outputs)
    self.b = tf.Variable(tf.zeros(n_outputs))

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    # The logits: the softmax is applied by the loss function
    return X @ self.W + self.b


simple_nn = SimpleNN()
print(
  simple_nn(prepare_images(X_train[:5])).shape,
  len(simple_nn.trainable_variables),
)

### Computing the Loss Function

The standard loss function in multi-class classification is cross-entropy. Keras provides it for integer labels (0 to 9, rather than one-hot vectors), in its so-called *sparse* version: `keras.losses.sparse_categorical_crossentropy`, with `from_logits=True` since the model returns logits. It returns one value per example: to be able to interpret it directly, we take its mean with `tf.reduce_mean`.

*Complete the `sparse_crossentropy` function. What value does it take for a model that hasn't learned yet?*

**Useful documentation**

- [`keras.losses.sparse_categorical_crossentropy`](https://keras.io/api/losses/probabilistic_losses/#sparse_categorical_crossentropy-function)

In [ ]:
def sparse_crossentropy(y: np.ndarray, logits: tf.Tensor) -> tf.Tensor:
  pass  # Your code here

#### Additional hints

- `keras.losses.sparse_categorical_crossentropy(y, logits, from_logits=True)`: the labels first, then the logits; one value per example
- `tf.reduce_mean(...)`: their mean, a scalar

A model that hasn't learned gives roughly the same probability, 1/10, to each class: its loss is about $-\log(1/10) = \log 10 \approx 2.3$.

```python
def sparse_crossentropy(y: np.ndarray, logits: tf.Tensor) -> tf.Tensor:
  cross_entropy = keras.losses.sparse_categorical_crossentropy(
    ..., ..., from_logits=...
  )
  return tf.reduce_mean(...)


print(sparse_crossentropy(y_train, SimpleNN()(prepare_images(X_train))))
```

#### Solution

In [ ]:
def sparse_crossentropy(y: np.ndarray, logits: tf.Tensor) -> tf.Tensor:
  # from_logits=True: the softmax is computed in the loss, in a stable way
  cross_entropy = keras.losses.sparse_categorical_crossentropy(
    y, logits, from_logits=True
  )
  return tf.reduce_mean(cross_entropy)


# About log(10) = 2.3: the model answers almost at random
print(sparse_crossentropy(y_train, SimpleNN()(prepare_images(X_train))))

### Metrics

To know whether a model learns, we measure its performance. We'll use the simplest metric, but also one of the most informative: accuracy, the number of right predictions over the total number of predictions.

*Complete the `accuracy` function. The predicted class is the one with the largest logit (`tf.argmax`): compare it with the labels, then compute the proportion of right answers.*

In [ ]:
def accuracy(y: np.ndarray, logits: tf.Tensor) -> float:
  pass  # Your code here

#### Additional hints

- `tf.argmax(logits, axis=1)`: the predicted class for each example (the softmax doesn't change the order of the scores, no need to compute it)
- `predictions == y`: a tensor of booleans
- `tf.cast(..., tf.float32)` then `tf.reduce_mean`: the proportion of `True`
- `float(...)`: a Python number, simpler to display and store

Expected: about 0.1 for a model that hasn't learned, one chance in ten.

```python
def accuracy(y: np.ndarray, logits: tf.Tensor) -> float:
  predictions = tf.argmax(..., axis=...)
  return float(tf.reduce_mean(tf.cast(..., tf.float32)))


print(accuracy(y_val, SimpleNN()(prepare_images(X_val))))
```

#### Solution

In [ ]:
def accuracy(y: np.ndarray, logits: tf.Tensor) -> float:
  predictions = tf.argmax(logits, axis=1)
  return float(tf.reduce_mean(tf.cast(predictions == y, tf.float32)))


print(accuracy(y_val, SimpleNN()(prepare_images(X_val))))

## Creating a TensorFlow Dataset

The model will learn by mini-batches: at each step, a small group of randomly drawn examples. `tf.data` prepares them, from the raw images.

*Complete the `create_dataset` function, which creates a `tf.data.Dataset` from the `X` and `y` arrays (`from_tensor_slices`), shuffles it at each epoch (`shuffle`), splits it into `(X_batch, y_batch)` batches of `batch_size` examples (`batch`), prepares the images of each batch with `prepare_images` (`map`), and prepares the next batch during the computation (`prefetch`). Why apply `map` after `batch` rather than before?*

**Useful documentation**

- [`tf.data.Dataset`](https://www.tensorflow.org/api_docs/python/tf/data/Dataset)

In [ ]:
def create_dataset(
  X: np.ndarray, y: np.ndarray, batch_size: int = 128
) -> tf.data.Dataset:
  pass  # Your code here

### Additional hints

- `tf.data.Dataset.from_tensor_slices((X, y))`: a dataset of `(x, y)` examples
- `.shuffle(len(X))` **before** `.batch(batch_size)`: we shuffle the examples; after, we would only shuffle the order of the same batches. The buffer of `len(X)` examples gives a full shuffle, redone at each epoch.
- `.map(lambda X_batch, y_batch: (prepare_images(X_batch), y_batch), num_parallel_calls=tf.data.AUTOTUNE)`: the function receives the two elements of each batch, and only transforms the images
- `.prefetch(tf.data.AUTOTUNE)`: last

Expected: `create_dataset(X_train, y_train)` has 391 batches (`len`), and the first one, `next(iter(...))`, has images of shape (128, 784), as `float32`.

```python
def create_dataset(
  X: np.ndarray, y: np.ndarray, batch_size: int = 128
) -> tf.data.Dataset:
  dataset = tf.data.Dataset.from_tensor_slices(...)
  dataset = dataset.shuffle(...)
  dataset = dataset.batch(...)
  dataset = dataset.map(
    lambda X_batch, y_batch: (..., y_batch), num_parallel_calls=tf.data.AUTOTUNE
  )
  return dataset.prefetch(...)


dataset = create_dataset(X_train, y_train)
X_batch, y_batch = next(iter(dataset))
print(len(dataset), X_batch.shape, X_batch.dtype, y_batch.shape)
```

### Solution

In [ ]:
def create_dataset(
  X: np.ndarray, y: np.ndarray, batch_size: int = 128
) -> tf.data.Dataset:
  dataset = tf.data.Dataset.from_tensor_slices((X, y))
  # Shuffling the examples, before splitting into batches, redone at each epoch
  dataset = dataset.shuffle(len(X))
  dataset = dataset.batch(batch_size)
  # Preparing the images, a whole batch at a time
  dataset = dataset.map(
    lambda X_batch, y_batch: (prepare_images(X_batch), y_batch),
    num_parallel_calls=tf.data.AUTOTUNE,
  )
  # Preparing the next batch during the computation on the current one
  return dataset.prefetch(tf.data.AUTOTUNE)


dataset = create_dataset(X_train, y_train)
X_batch, y_batch = next(iter(dataset))
print(len(dataset), X_batch.shape, X_batch.dtype, y_batch.shape)

After `batch`, `map` calls `prepare_images` once per batch, on 128 images at a time, rather than once per image: 391 calls per epoch instead of 50,000, and vectorized operations. `prepare_images` accepts a batch, of shape (128, 28, 28), like a whole set. Shuffling works on the raw images: its buffer of 50,000 images, as `uint8`, takes four times less memory than as `float32`.

## Training

Here is an almost complete training loop: it goes through the batches at each epoch, then measures the loss and the accuracy on the training and validation sets, and plots their curves at the end.

*Implement the missing part: updating the model's variables for one batch, by gradient descent. Compute the loss under a `tf.GradientTape`, adding `regularization(model)` to it when a regularization is given (it will be used later), then its gradients, and subtract `learning_rate` times its gradient from each variable with `assign_sub`.*

**Useful documentation**

- [`tf.GradientTape`](https://www.tensorflow.org/api_docs/python/tf/GradientTape)
- [`tf.Variable.assign_sub`](https://www.tensorflow.org/api_docs/python/tf/Variable#assign_sub)

In [ ]:
LossFunction = Callable[[np.ndarray, tf.Tensor], tf.Tensor]
RegularizationFunction = Callable[[tf.Module], tf.Tensor]


def train(
  model: tf.Module,
  X_train: np.ndarray = X_train,
  y_train: np.ndarray = y_train,
  X_val: np.ndarray = X_val,
  y_val: np.ndarray = y_val,
  epochs: int = 10,
  batch_size: int = 128,
  learning_rate: float = 0.1,
  evaluate_every: int = 1,
  loss: LossFunction = sparse_crossentropy,
  regularization: RegularizationFunction | None = None,
) -> tuple[list[float], ...]:
  # The whole sets, prepared once for the evaluations
  X_train_prep, X_val_prep = prepare_images(X_train), prepare_images(X_val)
  y_train_pred = model(X_train_prep)
  y_val_pred = model(X_val_prep)
  accuracies = [accuracy(y_train, y_train_pred)]
  val_accuracies = [accuracy(y_val, y_val_pred)]
  losses = [float(loss(y_train, y_train_pred))]
  val_losses = [float(loss(y_val, y_val_pred))]

  print(
    f"Initial metrics: acc {accuracies[-1]:.4f}, "
    f"val_acc {val_accuracies[-1]:.4f}, "
    f"loss {losses[-1]:.4f}, "
    f"val_loss {val_losses[-1]:.4f}"
  )

  dataset_train = create_dataset(X_train, y_train, batch_size)

  for epoch in range(epochs):
    for X_batch, y_batch in dataset_train:
      pass  # Your code here

    # Computing and displaying the metrics every `evaluate_every` epochs
    if (epoch + 1) % evaluate_every == 0:
      y_train_pred = model(X_train_prep)
      y_val_pred = model(X_val_prep)
      accuracies.append(accuracy(y_train, y_train_pred))
      losses.append(float(loss(y_train, y_train_pred)))
      val_accuracies.append(accuracy(y_val, y_val_pred))
      val_losses.append(float(loss(y_val, y_val_pred)))
      print(
        f"Epoch {epoch + 1}: acc {accuracies[-1]:.4f}, "
        f"val_acc {val_accuracies[-1]:.4f}, "
        f"loss {losses[-1]:.4f}, "
        f"val_loss {val_losses[-1]:.4f}"
      )

  x_ticks = np.arange(0, epochs + 1, evaluate_every)

  plt.plot(x_ticks, losses, label="Training")
  plt.plot(x_ticks, val_losses, label="Validation")
  plt.title("Loss function during training")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(x_ticks, accuracies, label="Training")
  plt.plot(x_ticks, val_accuracies, label="Validation")
  plt.title("Accuracy during training")
  plt.xlabel("Epochs")
  plt.legend(loc="lower right")
  plt.show()
  return accuracies, val_accuracies, losses, val_losses

### Additional hints

- `with tf.GradientTape() as tape:`: the operations of the block are recorded
- `loss(y_batch, model(X_batch))`: the loss of the batch; `+= regularization(model)` if `regularization is not None`
- `tape.gradient(loss_value, model.trainable_variables)`: the list of gradients, in the order of the variables
- `variable.assign_sub(learning_rate * gradient)`: the update, for each pair of `zip(gradients, model.trainable_variables)`

The completed loop over the batches, to copy in place of its own (with its indentation):

```python
for X_batch, y_batch in dataset_train:
  with tf.GradientTape() as tape:
    loss_value = ...
    if regularization is not None:
      loss_value += ...
  gradients = tape.gradient(..., model.trainable_variables)
  for gradient, variable in zip(gradients, model.trainable_variables):
    variable.assign_sub(...)
```

### Solution

In [ ]:
LossFunction = Callable[[np.ndarray, tf.Tensor], tf.Tensor]
RegularizationFunction = Callable[[tf.Module], tf.Tensor]


def train(
  model: tf.Module,
  X_train: np.ndarray = X_train,
  y_train: np.ndarray = y_train,
  X_val: np.ndarray = X_val,
  y_val: np.ndarray = y_val,
  epochs: int = 10,
  batch_size: int = 128,
  learning_rate: float = 0.1,
  evaluate_every: int = 1,
  loss: LossFunction = sparse_crossentropy,
  regularization: RegularizationFunction | None = None,
) -> tuple[list[float], ...]:
  # The whole sets, prepared once for the evaluations
  X_train_prep, X_val_prep = prepare_images(X_train), prepare_images(X_val)
  y_train_pred = model(X_train_prep)
  y_val_pred = model(X_val_prep)
  accuracies = [accuracy(y_train, y_train_pred)]
  val_accuracies = [accuracy(y_val, y_val_pred)]
  losses = [float(loss(y_train, y_train_pred))]
  val_losses = [float(loss(y_val, y_val_pred))]

  print(
    f"Initial metrics: acc {accuracies[-1]:.4f}, "
    f"val_acc {val_accuracies[-1]:.4f}, "
    f"loss {losses[-1]:.4f}, "
    f"val_loss {val_losses[-1]:.4f}"
  )

  dataset_train = create_dataset(X_train, y_train, batch_size)

  for epoch in range(epochs):
    for X_batch, y_batch in dataset_train:
      # Computing the loss, operations recorded by the tape
      with tf.GradientTape() as tape:
        loss_value = loss(y_batch, model(X_batch))
        if regularization is not None:
          loss_value += regularization(model)

      # Gradient of the loss with respect to each variable of the model
      gradients = tape.gradient(loss_value, model.trainable_variables)

      # Updating each variable according to the rule:
      # new_value = old_value - learning_rate * gradient
      for gradient, variable in zip(gradients, model.trainable_variables):
        variable.assign_sub(learning_rate * gradient)

    # Computing and displaying the metrics every `evaluate_every` epochs
    if (epoch + 1) % evaluate_every == 0:
      y_train_pred = model(X_train_prep)
      y_val_pred = model(X_val_prep)
      accuracies.append(accuracy(y_train, y_train_pred))
      losses.append(float(loss(y_train, y_train_pred)))
      val_accuracies.append(accuracy(y_val, y_val_pred))
      val_losses.append(float(loss(y_val, y_val_pred)))
      print(
        f"Epoch {epoch + 1}: acc {accuracies[-1]:.4f}, "
        f"val_acc {val_accuracies[-1]:.4f}, "
        f"loss {losses[-1]:.4f}, "
        f"val_loss {val_losses[-1]:.4f}"
      )

  x_ticks = np.arange(0, epochs + 1, evaluate_every)

  plt.plot(x_ticks, losses, label="Training")
  plt.plot(x_ticks, val_losses, label="Validation")
  plt.title("Loss function during training")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(x_ticks, accuracies, label="Training")
  plt.plot(x_ticks, val_accuracies, label="Validation")
  plt.title("Accuracy during training")
  plt.xlabel("Epochs")
  plt.legend(loc="lower right")
  plt.show()
  return accuracies, val_accuracies, losses, val_losses

### Training the Network Without a Hidden Layer

10 epochs, with a learning rate of 0.5:

In [ ]:
simple_nn = SimpleNN()
_ = train(simple_nn, epochs=10, learning_rate=0.5)

*What validation accuracy does this model reach? Does it still improve after a few epochs?*

Your answer here

#### Solution

About 92% validation accuracy (from 92.2 to 92.8% depending on the epoch). Most of it is learned in the first epoch (91.4%); after that, the model gains only one more point in 9 epochs, and oscillates a little. The training and validation accuracies stay equal: the model doesn't overfit, it is too simple. Logistic regression separates the classes with linear boundaries, which caps it at about 92% on MNIST.

## With a Keras Optimizer

The update written by hand above (`variable.assign_sub(learning_rate * gradient)`) is exactly what the `keras.optimizers.SGD` optimizer does. Keras optimizers work without a Keras model, on plain `tf.Variable`s: `optimizer.apply_gradients(zip(gradients, variables))` replaces the update loop.

*Complete `train_step`, one training step of `model` by `optimizer` on a batch: the loss of the batch under a `tf.GradientTape`, its gradients, then one optimizer step with `apply_gradients`; the function returns the loss. `train_with_optimizer`, which follows it, calls it for each batch; it will be used again later. The cell after the solution trains a `SimpleNN` with `keras.optimizers.SGD` and the same learning rate as before, then another one with `keras.optimizers.Adam`.*

**Useful documentation**

- [`keras.optimizers.SGD`](https://keras.io/api/optimizers/sgd/)
- [`apply_gradients`](https://keras.io/api/optimizers/#applygradients-method)

In [ ]:
def train_step(
  model: tf.Module,
  optimizer: keras.optimizers.Optimizer,
  X_batch: tf.Tensor,
  y_batch: tf.Tensor,
) -> tf.Tensor:
  pass  # Your code here


def train_with_optimizer(
  model: tf.Module,
  optimizer: keras.optimizers.Optimizer,
  epochs: int = 5,
  batch_size: int = 128,
) -> None:
  dataset_train = create_dataset(X_train, y_train, batch_size)
  X_val_prep = prepare_images(X_val)
  for epoch in range(epochs):
    for X_batch, y_batch in dataset_train:
      train_step(model, optimizer, X_batch, y_batch)

    val_accuracy = accuracy(y_val, model(X_val_prep))
    print(f"Epoch {epoch + 1}: val_acc {val_accuracy:.4f}")

### Additional hints

- The loss and the gradients are computed as in `train`, without regularization, with `sparse_crossentropy`
- `optimizer.apply_gradients(zip(gradients, model.trainable_variables))`: one optimizer step on all the variables

A Keras optimizer creates its own variables (Adam's averages, for instance) for the variables it sees at the first step: a new model needs a new optimizer.

```python
def train_step(
  model: tf.Module,
  optimizer: keras.optimizers.Optimizer,
  X_batch: tf.Tensor,
  y_batch: tf.Tensor,
) -> tf.Tensor:
  with tf.GradientTape() as tape:
    loss_value = sparse_crossentropy(..., ...)
  gradients = tape.gradient(..., ...)
  optimizer.apply_gradients(zip(..., ...))
  return loss_value
```

### Solution

In [ ]:
def train_step(
  model: tf.Module,
  optimizer: keras.optimizers.Optimizer,
  X_batch: tf.Tensor,
  y_batch: tf.Tensor,
) -> tf.Tensor:
  with tf.GradientTape() as tape:
    loss_value = sparse_crossentropy(y_batch, model(X_batch))
  gradients = tape.gradient(loss_value, model.trainable_variables)
  # One optimizer step, which replaces the assign_sub written by hand
  optimizer.apply_gradients(zip(gradients, model.trainable_variables))
  return loss_value

### Comparing SGD and Adam

5 epochs with each: SGD with the learning rate of the loop written by hand, Adam with its usual learning rate, much smaller.

In [ ]:
print("SGD")
simple_nn = SimpleNN()
train_with_optimizer(simple_nn, keras.optimizers.SGD(learning_rate=0.5))

print("Adam")
simple_nn = SimpleNN()
train_with_optimizer(simple_nn, keras.optimizers.Adam(learning_rate=1e-3))

*Do you get the performance of the loop written by hand with SGD? What does Adam change?*

Your answer here

#### Solution

With SGD and the same learning rate, we get the performance of the loop written by hand (about 92% in 5 epochs): `apply_gradients` applies the same update rule. Changing the optimization algorithm now only takes changing the optimizer. Adam adapts the step of each variable to the history of its gradients: with a learning rate 500 times smaller, it reaches the same result. On this simple model, it brings nothing more; its advantage is that the value 10⁻³ suits most networks, whereas SGD's has to be searched for each one.

## Compiling the Training Step With `tf.function`

So far, TensorFlow runs each operation as soon as Python calls it (*eager* mode). Decorated with `tf.function`, a function is first *traced*: its Python code runs once, and TensorFlow records its operations in a graph, which it then runs without going back through Python. A call whose arguments have another *signature* (shape or `dtype` of a tensor, value of a Python argument) triggers a new tracing.

The next cell defines `step`, one SGD training step of a new `SimpleNN`, which counts how many times its Python code runs, and `time_epoch(step_function, name)`, which times one epoch done with `step_function` and displays that count. It times one epoch in eager mode.

1. *Create `compiled_step`, the version of `step` compiled by `tf.function`, and time two epochs with it. Which of the two versions is the faster, and why? How many times does the Python code of `step` run at each epoch, and why?*
2. *Create `signature_step`, compiled with an `input_signature` that accepts batches of any size, and time one epoch with it. How many tracings are left?*

**Useful documentation**

- [`tf.function`](https://www.tensorflow.org/api_docs/python/tf/function)
- [`tf.TensorSpec`](https://www.tensorflow.org/api_docs/python/tf/TensorSpec)

In [ ]:
model = SimpleNN()
optimizer = keras.optimizers.SGD(learning_rate=0.5)
dataset_train = create_dataset(X_train, y_train)
n_runs = 0


def step(X_batch: tf.Tensor, y_batch: tf.Tensor) -> tf.Tensor:
  global n_runs
  # Python code: runs at each call in eager mode, at the tracings only once
  # compiled
  n_runs += 1
  return train_step(model, optimizer, X_batch, y_batch)


def time_epoch(
  step_function: Callable[[tf.Tensor, tf.Tensor], tf.Tensor], name: str
) -> None:
  global n_runs
  n_runs = 0
  start = time.perf_counter()
  for X_batch, y_batch in dataset_train:
    step_function(X_batch, y_batch)
  duration = time.perf_counter() - start
  print(f"{name}: {duration:.2f} s, Python code of step run {n_runs} times")


time_epoch(step, "Eager")

In [ ]:
# Your code here

### Additional hints

- `tf.function(step)`: the compiled version, to call like `step`
- `time_epoch(compiled_step, "Compiled, epoch 1")`: the duration of one epoch with it, and the number of runs of the Python code
- `tf.TensorSpec(shape=(None, 784), dtype=tf.float32)`: a tensor of shape `(n, 784)`, whatever `n`, like the images of a batch; the labels of a batch are `tf.uint8` of shape `(n,)`
- `tf.function(step, input_signature=[...])`: one specification per argument, in order

```python
compiled_step = tf.function(...)
time_epoch(compiled_step, "Compiled, epoch 1")
time_epoch(..., "Compiled, epoch 2")

signature_step = tf.function(
  step,
  input_signature=[
    tf.TensorSpec(shape=(None, ...), dtype=...),
    tf.TensorSpec(shape=..., dtype=...),
  ],
)
time_epoch(..., "With input_signature")
```

### Solution

In [ ]:
compiled_step = tf.function(step)
# The first call of each batch shape traces the function
time_epoch(compiled_step, "Compiled, epoch 1")
time_epoch(compiled_step, "Compiled, epoch 2")

# A single signature, for batches of any size
signature_step = tf.function(
  step,
  input_signature=[
    tf.TensorSpec(shape=(None, 28 * 28), dtype=tf.float32),
    tf.TensorSpec(shape=(None,), dtype=tf.uint8),
  ],
)
time_epoch(signature_step, "With input_signature")

The compiled version is 2 to 4 times faster, depending on the machine and its load: the graph runs without going back through Python at each operation, and TensorFlow can optimize it (operation fusion). The smaller the operations, the larger the gain: here, Python's overhead weighs heavily against matrix products of 128 examples.

In eager mode, the Python code of `step` runs at each step: 391 times per epoch. Compiled, `step` only runs it at its tracings: 2 times in the first epoch, because the last batch, of 80 examples (50,000 = 390 $\times$ 128 + 80), has another shape than the others and triggers a second tracing; 0 times in the second, both graphs being cached. A `print` in a compiled function does the same: it displays nothing more after the tracings, whereas `tf.print` runs at each call.

With `input_signature`, the first dimension is free (`None`): a single graph serves every batch size, and `signature_step` is traced only once. A tracing only costs a fraction of a second; on the other hand, a function called with ever new shapes would be retraced at each call, and be slower than in eager mode. `drop_remainder=True` in `batch` would also avoid the second tracing, at the cost of the last 80 examples of each epoch.

## Moving On to Deep Networks

To move on to a multilayer perceptron, we create one weight matrix and one bias vector per layer. The following function stores them in a list, one `{"W": ..., "b": ...}` dictionary per layer:

In [ ]:
def create_mlp_weights(
  n_inputs: int, n_outputs: int, hidden_layer_sizes: tuple[int, ...] = (512,)
) -> list[dict[str, tf.Variable]]:
  layers = []
  # A layer's input size is the previous one's output size
  last_size = n_inputs
  for size in [*hidden_layer_sizes, n_outputs]:
    layers.append(
      {"W": init_weights(last_size, size), "b": tf.Variable(tf.zeros(size))}
    )
    last_size = size
  return layers

The model's output is now computed layer by layer: each layer takes the previous one's result as input.

*Complete `MLP.__call__`. Each hidden layer computes $\text{ReLU}(XW + b)$ with `tf.nn.relu`; the last one returns the logits $XW + b$, without an activation. The variables of layer `i` are `self.layers[i]["W"]` and `self.layers[i]["b"]`.*

In [ ]:
class MLP(tf.Module):
  def __init__(
    self,
    hidden_layer_sizes: tuple[int, ...] = (512,),
    n_inputs: int = 28**2,
    n_outputs: int = 10,
  ) -> None:
    super().__init__()
    self.layers = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    pass  # Your code here

### Additional hints

- `self.layers[:-1]`: the hidden layers; `self.layers[-1]`: the output layer
- `tf.nn.relu(X @ layer["W"] + layer["b"])`: a hidden layer

Expected: `MLP()(prepare_images(X_train[:5]))` has shape (5, 10), and `MLP((256, 128))` has 6 variables (`trainable_variables`).

```python
class MLP(tf.Module):
  def __init__(
    self,
    hidden_layer_sizes: tuple[int, ...] = (512,),
    n_inputs: int = 28**2,
    n_outputs: int = 10,
  ) -> None:
    super().__init__()
    self.layers = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    for layer in self.layers[:-1]:
      X = tf.nn.relu(...)
    output_layer = self.layers[-1]
    return ...
```

### Solution

In [ ]:
class MLP(tf.Module):
  def __init__(
    self,
    hidden_layer_sizes: tuple[int, ...] = (512,),
    n_inputs: int = 28**2,
    n_outputs: int = 10,
  ) -> None:
    super().__init__()
    self.layers = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    # Hidden layers: affine transformation, then ReLU
    for layer in self.layers[:-1]:
      X = tf.nn.relu(X @ layer["W"] + layer["b"])
    # Output layer: the logits, without an activation
    output_layer = self.layers[-1]
    return X @ output_layer["W"] + output_layer["b"]


print(
  MLP()(prepare_images(X_train[:5])).shape, len(MLP((256, 128)).trainable_variables)
)

## Training the Deep Network

One hidden layer of 512 neurons, 10 epochs, a learning rate of 0.5:

In [ ]:
mlp = MLP()
_ = train(mlp, epochs=10, learning_rate=0.5)

*What do you notice when comparing this network with the network without a hidden layer?*

Your answer here

### Solution

The network with one hidden layer does much better than the network without a hidden layer: about 98% validation accuracy, against 92%, an error divided by 4. The hidden layer and its ReLU make the model nonlinear. The training accuracy goes above 99% and moves away from the validation one: the network starts to overfit, but its validation loss is still decreasing.

This is partly thanks to the weight initialization, with a variance of $1 / n_\text{in}$. To convince yourself, remove the division by `n_in**0.5` in `init_weights` and run again: the initial logits have a standard deviation of the order of 100, the initial loss is in the hundreds instead of 2.3, and the network stays far behind after 2 epochs.

## Regularization

The previous MLP overfits little: its validation loss is still decreasing at the tenth epoch, since 50,000 examples are enough for its 400,000 weights. Overfitting shows when the network is large compared to the data: the cell after the solution trains the same MLP on the first 1,000 examples only, for 100 epochs, without then with regularization.

L2 regularization adds a penalty to the loss, $\lambda \sum w^2$: $\lambda$ times the sum of the squares of the weights, which pushes the weights toward 0. Only the `W` weights are penalized, not the biases: they don't multiply the input, and constraining them doesn't help to generalize.

*Complete the inner function of `l2`, which returns the penalty of an `MLP`: the sum of the squares of the `W` matrices of all its layers (`model.layers`), multiplied by `lambda_coefficient`.*

**Useful documentation**

- [`tf.reduce_sum`](https://www.tensorflow.org/api_docs/python/tf/math/reduce_sum)

In [ ]:
def l2(lambda_coefficient: float) -> RegularizationFunction:
  def penalty(model: MLP) -> tf.Tensor:
    pass  # Your code here

  return penalty

### Additional hints

- `tf.reduce_sum(layer["W"] ** 2)`: the sum of the squares of a matrix
- Python's `sum(...)`, or `tf.add_n([...])`, to add those of all the layers

Expected: for a new `MLP()`, `l2(1.0)(MLP())` is about 522 (512 for the first layer, 10 for the second: each one contributes roughly its number of columns, since each weight has a variance of $1 / n_\text{in}$).

```python
def l2(lambda_coefficient: float) -> RegularizationFunction:
  def penalty(model: MLP) -> tf.Tensor:
    weights_norm = sum(tf.reduce_sum(...) for layer in model.layers)
    return ... * weights_norm

  return penalty
```

### Solution

In [ ]:
def l2(lambda_coefficient: float) -> RegularizationFunction:
  def penalty(model: MLP) -> tf.Tensor:
    # The W weights only, not the biases
    weights_norm = sum(tf.reduce_sum(layer["W"] ** 2) for layer in model.layers)
    return lambda_coefficient * weights_norm

  return penalty


print(float(l2(1.0)(MLP())))

### Overfitting on 1,000 Examples

The same MLP, with the same learning rate of 0.5, on the first 1,000 training examples, without regularization then with an L2 penalty of $\lambda$ = 10⁻³; validation is still done on the 10,000 examples set aside.

In [ ]:
X_small, y_small = X_train[:1000], y_train[:1000]

print("Without regularization")
_ = train(
  MLP(),
  X_train=X_small,
  y_train=y_small,
  epochs=100,
  learning_rate=0.5,
  evaluate_every=10,
)

print("With an L2 penalty")
_ = train(
  MLP(),
  X_train=X_small,
  y_train=y_small,
  epochs=100,
  learning_rate=0.5,
  evaluate_every=10,
  regularization=l2(1e-3),
)

*Compare the two training runs: the gap between training and validation, the validation loss over the epochs, the final validation accuracy.*

Your answer here

#### Solution

Without regularization, the network learns the 1,000 examples by heart: 100% training accuracy from the twentieth epoch on, a training loss that goes to 0, and a validation accuracy stuck at about 89.5%. The validation loss reaches a minimum around the twentieth epoch, then goes back up: the network becomes more and more confident in its answers, wrong ones included. That's overfitting.

With the L2 penalty, the training loss stays above 0.02 (the weights can no longer grow freely) and the validation loss no longer goes back up. The validation accuracy gains little, a few tenths of a point: on so little data, regularization mostly limits the model's overconfidence. Nothing replaces more data: the same network reaches 98% with the 50,000 examples.

## Following Training With TensorBoard

In the Keras lab, the `keras.callbacks.TensorBoard` callback wrote the metrics of `fit` at each epoch to a log, which TensorBoard displays. Without `fit`, we write that log ourselves:

- `tf.summary.create_file_writer(log_dir)` creates a log in `log_dir`;
- in a `with writer.as_default():` block, `tf.summary.scalar(name, value, step=epoch)` adds a point to the `name` curve.

One directory per run: TensorBoard overlays their curves, which lets us compare hyperparameters on the validation set, never on the test set.

*Complete `train_with_log`: at each epoch, take one training step per batch with `train_step`, compiled by `tf.function`, then write to the log the mean loss of the epoch (`loss`) and the validation accuracy (`val_accuracy`). The function returns the last validation accuracy. The cell after the solution trains an MLP by SGD with three learning rates, each in its own directory of `logs/`, then opens TensorBoard.*

**Useful documentation**

- [`tf.summary.create_file_writer`](https://www.tensorflow.org/api_docs/python/tf/summary/create_file_writer)
- [`tf.summary.scalar`](https://www.tensorflow.org/api_docs/python/tf/summary/scalar)

In [ ]:
def train_with_log(
  model: tf.Module,
  optimizer: keras.optimizers.Optimizer,
  log_dir: str,
  epochs: int = 5,
) -> float:
  pass  # Your code here

### Additional hints

- A compiled step, specific to this model and this optimizer, defined in the function:

```python
@tf.function
def step(X_batch: tf.Tensor, y_batch: tf.Tensor) -> tf.Tensor:
  return train_step(model, optimizer, X_batch, y_batch)
```

- `tf.reduce_mean(losses)`: the mean loss of the epoch, from the list of the losses of its batches
- `accuracy(y_val, model(X_val_prep))`, with `X_val_prep = prepare_images(X_val)` prepared once
- `step=epoch + 1`: the point's abscissa, in epochs

```python
def train_with_log(
  model: tf.Module,
  optimizer: keras.optimizers.Optimizer,
  log_dir: str,
  epochs: int = 5,
) -> float:
  writer = tf.summary.create_file_writer(...)

  @tf.function
  def step(X_batch: tf.Tensor, y_batch: tf.Tensor) -> tf.Tensor:
    return train_step(model, optimizer, X_batch, y_batch)

  dataset_train = create_dataset(X_train, y_train)
  X_val_prep = prepare_images(X_val)
  for epoch in range(epochs):
    losses = [step(X_batch, y_batch) for X_batch, y_batch in dataset_train]
    val_accuracy = accuracy(y_val, model(X_val_prep))
    with writer.as_default():
      tf.summary.scalar("loss", ..., step=epoch + 1)
      tf.summary.scalar(..., ..., step=...)
  return val_accuracy
```

### Solution

In [ ]:
def train_with_log(
  model: tf.Module,
  optimizer: keras.optimizers.Optimizer,
  log_dir: str,
  epochs: int = 5,
) -> float:
  writer = tf.summary.create_file_writer(log_dir)

  # A compiled step specific to this model and this optimizer
  @tf.function
  def step(X_batch: tf.Tensor, y_batch: tf.Tensor) -> tf.Tensor:
    return train_step(model, optimizer, X_batch, y_batch)

  dataset_train = create_dataset(X_train, y_train)
  X_val_prep = prepare_images(X_val)
  for epoch in range(epochs):
    losses = [step(X_batch, y_batch) for X_batch, y_batch in dataset_train]
    val_accuracy = accuracy(y_val, model(X_val_prep))
    # One point per curve and per epoch
    with writer.as_default():
      tf.summary.scalar("loss", tf.reduce_mean(losses), step=epoch + 1)
      tf.summary.scalar("val_accuracy", val_accuracy, step=epoch + 1)
    print(f"Epoch {epoch + 1}: val_acc {val_accuracy:.4f}")
  return val_accuracy

### Comparing Three Learning Rates

One MLP per learning rate, 5 epochs each, logged to `logs/sgd_<learning rate>`:

In [ ]:
mlps, val_accuracies = {}, {}
for rate in (0.01, 0.1, 0.5):
  print(f"Learning rate {rate}")
  mlps[rate] = MLP()
  optimizer = keras.optimizers.SGD(learning_rate=rate)
  val_accuracies[rate] = train_with_log(mlps[rate], optimizer, f"logs/sgd_{rate}")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

*In TensorBoard, which learning rate gives the best validation accuracy? What do the loss curves show?*

Your answer here

#### Solution

0.5: about 97.7% validation accuracy in 5 epochs, against 95.7% for 0.1 and 91% for 0.01 (`val_accuracy` curve). The `loss` curves show why: the smaller the learning rate, the more slowly the training loss goes down. At 0.01, the network is far from done learning in 5 epochs: comparing such short runs favors what learns fast, not necessarily what ends best.

TensorBoard reads the logs again as they are written: you can follow a training run in progress. Running the cell again adds new logs to the old ones, in the same directories: delete `logs/` to start from scratch.

## Final Evaluation on the Test Set

The test set hasn't been used so far: neither for training, nor for choosing the learning rate. That's what makes its evaluation reliable.

*Take `mlp`, the MLP trained with the learning rate of the best validation accuracy, and compute its accuracy on `X_test`. Is it close to its validation accuracy?*

In [ ]:
# Your code here

### Additional hints

- `max(val_accuracies, key=val_accuracies.get)`: the key of the dictionary's largest value, here the best learning rate
- `mlps[best_rate]`: the MLP trained with it
- `accuracy(y_test, mlp(prepare_images(X_test)))`: the test accuracy

```python
best_rate = max(..., key=...)
mlp = mlps[...]
print(f"Accuracy on the test set: {accuracy(..., ...):.4f}")
```

### Solution

In [ ]:
# The learning rate of the best validation accuracy, and its MLP
best_rate = max(val_accuracies, key=val_accuracies.get)
mlp = mlps[best_rate]
print(f"Chosen learning rate: {best_rate}")
print(f"Accuracy on the validation set: {val_accuracies[best_rate]:.4f}")
print(f"Accuracy on the test set: {accuracy(y_test, mlp(prepare_images(X_test))):.4f}")

The test accuracy is close to the validation accuracy (here 97.7% for both). The validation set was used to choose the learning rate: its accuracy is in principle slightly optimistic, all the more so as more runs were compared. The test set, which wasn't used for any choice, gives a reliable estimate of the model's performance on new data.

## Saving and Restoring the Model

Training is costly: we save the trained model to reuse it without training it again. `tf.train.Checkpoint` records the values of an object's variables, found again by their attribute names.

*Save `mlp` to `checkpoints/mlp`. Create `restored`, a new `MLP` with the same architecture, and display its test accuracy before and after restoring the checkpoint into it.*

**Useful documentation**

- [`tf.train.Checkpoint`](https://www.tensorflow.org/api_docs/python/tf/train/Checkpoint)

In [ ]:
# Your code here

### Additional hints

- `tf.train.Checkpoint(model=mlp).save("checkpoints/mlp")`: the saving; it returns the checkpoint's path, `checkpoints/mlp-1` (the number goes up at each save)
- `tf.train.Checkpoint(model=restored).restore(path)`: the restoring; the argument name, `model`, is free, but must be the same as when saving
- `restored` must have the same architecture as `mlp`: an `MLP()`, with its default sizes

```python
path = tf.train.Checkpoint(model=...).save("checkpoints/mlp")
restored = MLP()
X_test_prep = prepare_images(X_test)
print(accuracy(y_test, restored(X_test_prep)))
tf.train.Checkpoint(model=...).restore(...)
print(accuracy(..., ...))
```

### Solution

In [ ]:
path = tf.train.Checkpoint(model=mlp).save("checkpoints/mlp")
print(path)

# New random weights, same architecture as mlp
restored = MLP()
X_test_prep = prepare_images(X_test)
print(f"Before restoring: {accuracy(y_test, restored(X_test_prep)):.4f}")
tf.train.Checkpoint(model=restored).restore(path)
print(f"After restoring: {accuracy(y_test, restored(X_test_prep)):.4f}")
print(f"Saved model: {accuracy(y_test, mlp(X_test_prep)):.4f}")

Before restoring, `restored` answers almost at random (around 10%, its weights are random). After, it has exactly the accuracy of `mlp`: the checkpoint gave each variable back its value, found by its attribute path (`model`, then `layers`, then the index and the key). Hence the importance of an identical architecture: an `MLP` of another size would have variables of different shapes, and the restoration would fail.

A checkpoint only holds values, not the model's code: to restore it, you need the `MLP` class. To deploy a model without its Python code, TensorFlow has another format, the SavedModel (`tf.saved_model.save`), which keeps the graph of the methods compiled by `tf.function`.